# Vectorized QMC

Demonstrates QMC integration with vectorized (batched) function evaluation.

Original QMCPy demo: [`QMCPy/demos/vectorized_qmc.ipynb`](../../QMCPy/demos/vectorized_qmc.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/vectorized_qmc.ipynb)

*This Julia notebook focuses on the vectorized integration pieces that have direct Julia counterparts. The QMCPy notebook also includes Bayesian optimization and machine-learning examples that are split into other Julia demos or require additional packages.*

In [1]:
using QMC
using LinearAlgebra: Diagonal
import QMC: Uniform
using Statistics

## LD Sequence

Compare IID, digital net, and lattice point sets.

In [2]:
n = 2^6
for (dd, name) in [
    (IIDStdUniform(2; seed=7),  "IID"),
    (DigitalNetB2(2; seed=7),   "Digital Net"),
    (Lattice(2; seed=7),        "Lattice"),
]
    pts = gen_samples(dd, n)
    println("$name: $(size(pts, 1)) points, " *
            "mean=$(round.(mean(pts, dims=1), digits=3))")
end

IID: 64 points, mean=[0.506 0.422]


Digital Net: 64 points, mean=[0.497 0.503]


Lattice: 64 points, mean=[0.507 0.499]


## Simple Example

The cantilever beam function maps 3-dimensional input to a 2-component output (displacement and stress). QMC.jl naturally handles batched evaluation of such vector-valued outputs.

In [3]:
# Cantilever beam: (E, X, Y) → (displacement D, stress S)
function cantilever_beam(x)
    l, w, t = 100.0, 4.0, 2.0
    E, X, Y = x[1], x[2], x[3]
    D = 4l^3 / (E * w * t) * sqrt(X^2 / t^4 + Y^2 / w^4)
    S = 600 * (X / (w * t^2) + Y / (w^2 * t))
    return [D, S]
end

# Match the QMCPy example: Gaussian input model for (E, X, Y)
dd = DigitalNetB2(3; seed=7, graycode=false)
tm = Gaussian(dd;
    mean=[2.9e7, 500.0, 1000.0],
    covariance=Diagonal([(1.45e6)^2, (100.0)^2, (100.0)^2]))
u = gen_samples(dd, 1024)
x = transform(tm, u)

results = hcat([cantilever_beam(x[i, :]) for i in 1:size(x, 1)]...)
println("Displacement: mean=$(round(mean(results[1,:]), digits=4)), " *
        "std=$(round(std(results[1,:]), digits=4))")
println("Stress:       mean=$(round(mean(results[2,:]), digits=4)), " *
        "std=$(round(std(results[2,:]), digits=4))")

Displacement: mean=2.425, std=0.4047


Stress:       mean=37489.9412, std=4194.3089


## Sensitivity Indices / Vector Outputs

Use `CubMCCLTVec` to integrate all output components simultaneously. This is the core vectorized capability highlighted by the QMCPy notebook.

The Python demo also includes sections on BO QEI, Bayesian logistic regression, Ishigami sensitivity indices, and a neural-network example. Those examples depend on additional Python packages and APIs that do not have direct one-to-one Julia notebook counterparts here, so this Julia version focuses on the vectorized cubature behavior itself.

In [4]:
# Wrap as CustomFun for integration
cf = CustomFun(tm, x -> cantilever_beam(x), 2)
sc = CubMCCLTVec(cf; abs_tol=0.1)
result = integrate(sc)
println("Solution: $(result.solution)")
println("Converged: $(result.data[:converged])")

Solution: 35.03124264574422


Converged: false
